# 🌒 EX0005C · Preflight laboratory, v0.3.1 (API compatibility patch)

**NOT A LIVE LUNA SURGERY.** This is the source-pinned, heldout-sealing and G0 qualification notebook. It **never executes a held-out model evaluation** and does not call the Luna API. Run in a Google Colab **L4 GPU** session. Stop on any failed assertion. No fallback model, token cap, or dataset substitutions.

The separate protocol `EX0005C_PROTOCOL_v0.3.md` defines all proposed rules. No claim of G0 PASS is made before running and inspecting these cells.

**Hard time boundary:** Sunday, November 22, 2026, by 20:00 America/Denver; no make-up run is authorized by this notebook.
**v0.3.1 technical erratum:** item-local sampling now uses `torch.random.fork_rng` + `torch.manual_seed` instead of the unsupported `generate(generator=...)` kwarg. No other candidate limits, sampling settings, data, or gates were changed.


In [ ]:
# Dependencies. Use Colab's existing CUDA PyTorch; pin tokenizer/data APIs.
%pip -q install 'transformers==4.57.3' 'datasets==3.6.0' 'accelerate>=1.0,<2' 'safetensors>=0.5,<1'
import torch,transformers,datasets,platform,json,datetime
from zoneinfo import ZoneInfo
print('Versions',{'torch':torch.__version__,'transformers':transformers.__version__,'datasets':datasets.__version__})
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'STOP: no CUDA GPU'
assert 'L4' in torch.cuda.get_device_name(0), 'STOP: not the specified Colab L4'
assert torch.cuda.is_bf16_supported(), 'STOP: BF16 unsupported'
assert datetime.datetime.now(ZoneInfo('America/Denver')).date().isoformat()=='2026-11-22', 'STOP: wrong night'
assert datetime.datetime.now(ZoneInfo('America/Denver')).hour < 20, 'STOP: Sunday atelier has ended'


In [ ]:
# Embedded, SHA-checked source snapshots: no GitHub file fetching needed.
from pathlib import Path
import base64,hashlib,os,subprocess,sys
snapshots={'ex0005c_scorer.py': {'source_b64': 'IiIiRVgwMDA1QyB2MC4zIHByZWZsaWdodCBzY29yZXIuIE9mZmxpbmUgYW5kIHNpZGUtZWZmZWN0IGZyZWUuCgpEbyBub3QgY291bnQgdHJ1bmNhdGVkLCBtYWxmb3JtZWQsIG9yIHVucGFyc2VhYmxlIHJlc3BvbnNlcyBhcyBjb3JyZWN0LgpUaGUgZXZhbHVhdG9yIHJlY29yZHMgJ2ZpbmlzaGVkJyBhbmQgJ2NvbXBsZXRlZCcgc2VwYXJhdGVseS4KIiIiCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgYXNkaWN0CmZyb20gZGVjaW1hbCBpbXBvcnQgRGVjaW1hbCwgSW52YWxpZE9wZXJhdGlvbgppbXBvcnQgcmUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgppbXBvcnQgcmFuZG9tCgpGSU5BTCA9IHJlLmNvbXBpbGUociIoP20pXkZpbmFsIGFuc3dlcjpbIFx0XSooW15cclxuXSo/KVsgXHRdKiQiKQpOVU0gPSByZS5jb21waWxlKHIiWystXT8oPzpcZHsxLDN9KD86LFxkezN9KSt8XGQrKSg/OlwuXGQrKT8iKQoKCmRlZiBudW1lcmljKHZhbHVlKToKICAgICIiIkFjY2VwdCBvbmx5IHNpZ25lZCBudW1lcmljIHNjYWxhcnMgd2l0aCBvcHRpb25hbCBjb3JyZWN0bHkgcGxhY2VkIGNvbW1hcy4iIiIKICAgIHZhbHVlID0gdmFsdWUuc3RyaXAoKQogICAgaWYgbm90IE5VTS5mdWxsbWF0Y2godmFsdWUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJpbnZhbGlkIG51bWVyaWMgYW5zd2VyOiB7dmFsdWUhcn0iKQogICAgdHJ5OgogICAgICAgIGQgPSBEZWNpbWFsKHZhbHVlLnJlcGxhY2UoJywnLCAnJykpCiAgICBleGNlcHQgSW52YWxpZE9wZXJhdGlvbiBhcyBleGM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBkZWNpbWFsJykgZnJvbSBleGMKICAgIGlmIG5vdCBkLmlzX2Zpbml0ZSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ25vbmZpbml0ZSBkZWNpbWFsJykKICAgIHJldHVybiBkCgoKZGVmIHJlZmVyZW5jZShhbnN3ZXIpOgogICAgIiIiRXh0cmFjdCBHU004SydzIGF1dGhvcml0YXRpdmUgIyMjIyBmaW5hbCBudW1lcmljIGFuc3dlci4iIiIKICAgIGhpdHMgPSByZS5maW5kYWxsKHIiKD9tKV4jIyMjXHMqKFteXHJcbl0rKVxzKiQiLCBhbnN3ZXIpCiAgICBpZiBsZW4oaGl0cykgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdleHBlY3RlZCBleGFjdGx5IG9uZSAjIyMjIHJlZmVyZW5jZSBhbnN3ZXInKQogICAgcmV0dXJuIG51bWVyaWMoaGl0c1swXSkKCgpAZGF0YWNsYXNzCmNsYXNzIFNjb3JlOgogICAgZmluaXNoZWQ6IGJvb2wKICAgIGNvbXBsZXRlZDogYm9vbAogICAgY29ycmVjdDogYm9vbAogICAgcGFyc2VkOiBzdHIgfCBOb25lCiAgICBzdGF0dXM6IHN0cgoKICAgIGRlZiBqc29uKHNlbGYpOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpkZWYgc2NvcmVfbWF0aChvdXRwdXQsIHRydXRoLCAqLCBmaW5pc2hlZCwgcmVxdWlyZV90aGluaz1UcnVlKToKICAgICIiIm91dHB1dCBtdXN0IGJlIHJhdyBnZW5lcmF0aW9uIHRleHQgKGV4Y2x1ZGluZyBwcm9tcHQpOyBmaW5pc2hlZCBpcyBFT1Mgc3RvcCwgbm90IGxlbmd0aCBjdXRvZmYuCgogICAgT25seSB0aGUgdGV4dCBhZnRlciB0aGUgY2xvc2VkIHRoaW5raW5nIHRhZyBjb3VudHMgYXMgYSBmaW5hbCBhbnN3ZXIuIE5vIGV4dHJhCiAgICBwcm9zZSBtYXkgZm9sbG93IEZpbmFsIGFuc3dlcjogLi4uIC4gVGhlIG1vZGVsIG5lZWQgbm90IHNob3cgY29ycmVjdCByZWFzb25pbmcuCiAgICAiIiIKICAgIGlmIG5vdCBmaW5pc2hlZDoKICAgICAgICByZXR1cm4gU2NvcmUoRmFsc2UsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ3RydW5jYXRlZF9vcl91bnRlcm1pbmF0ZWQnKQogICAgaWYgcmVxdWlyZV90aGluazoKICAgICAgICBpZiBvdXRwdXQuY291bnQoJzwvdGhpbms+JykgIT0gMSBvciBvdXRwdXQuY291bnQoJzx0aGluaz4nKSAhPSAxIG9yIG91dHB1dC5pbmRleCgnPHRoaW5rPicpID4gb3V0cHV0LmluZGV4KCc8L3RoaW5rPicpOgogICAgICAgICAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAndW5jbG9zZWRfb3JfbXVsdGlwbGVfdGhpbmtpbmdfYmxvY2tzJykKICAgICAgICBhbnN3ZXJfcGFydCA9IG91dHB1dC5zcGxpdCgnPC90aGluaz4nLCAxKVsxXQogICAgZWxzZToKICAgICAgICBhbnN3ZXJfcGFydCA9IG91dHB1dAogICAgYW5zd2VyX3BhcnQgPSBhbnN3ZXJfcGFydC5yZXBsYWNlKCc8fGltX2VuZHw+JywgJycpLnN0cmlwKCkKICAgIG1hdGNoZXMgPSBsaXN0KEZJTkFMLmZpbmRpdGVyKGFuc3dlcl9wYXJ0KSkKICAgIGlmIGxlbihtYXRjaGVzKSAhPSAxOgogICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICdtaXNzaW5nX29yX211bHRpcGxlX2ZpbmFsX2Fuc3dlcnMnKQogICAgbWF0Y2ggPSBtYXRjaGVzWzBdCiAgICBpZiBhbnN3ZXJfcGFydFttYXRjaC5lbmQoKTpdLnN0cmlwKCk6CiAgICAgICAgcmV0dXJuIFNjb3JlKFRydWUsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ3RyYWlsaW5nX3RleHRfYWZ0ZXJfZmluYWwnKQogICAgdHJ5OgogICAgICAgIGdvdCA9IG51bWVyaWMobWF0Y2guZ3JvdXAoMSkpCiAgICAgICAgZ29sZCA9IG51bWVyaWMoc3RyKHRydXRoKSkKICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICdpbnZhbGlkX251bWJlcicpCiAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgVHJ1ZSwgZ290ID09IGdvbGQsIHN0cihnb3QpLCAndmFsaWRfY29tcGxldGUnKQoKCmRlZiBzY29yZV90ZXh0KG91dHB1dCwgdHJ1dGgsICosIGZpbmlzaGVkLCByZXF1aXJlX3RoaW5rPVRydWUpOgogICAgaWYgbm90IGZpbmlzaGVkOgogICAgICAgIHJldHVybiBTY29yZShGYWxzZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAndHJ1bmNhdGVkX29yX3VudGVybWluYXRlZCcpCiAgICBpZiByZXF1aXJlX3RoaW5rOgogICAgICAgIGlmIG91dHB1dC5jb3VudCgnPC90aGluaz4nKSAhPSAxIG9yIG91dHB1dC5jb3VudCgnPHRoaW5rPicpICE9IDEgb3Igb3V0cHV0LmluZGV4KCc8dGhpbms+JykgPiBvdXRwdXQuaW5kZXgoJzwvdGhpbms+Jyk6CiAgICAgICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICd1bmNsb3NlZF9vcl9tdWx0aXBsZV90aGlua2luZ19ibG9ja3MnKQogICAgICAgIGFuc3dlcl9wYXJ0ID0gb3V0cHV0LnNwbGl0KCc8L3RoaW5rPicsIDEpWzFdCiAgICBlbHNlOgogICAgICAgIGFuc3dlcl9wYXJ0ID0gb3V0cHV0CiAgICBhbnN3ZXJfcGFydCA9IGFuc3dlcl9wYXJ0LnJlcGxhY2UoJzx8aW1fZW5kfD4nLCAnJykuc3RyaXAoKQogICAgbWF0Y2hlcyA9IGxpc3QoRklOQUwuZmluZGl0ZXIoYW5zd2VyX3BhcnQpKQogICAgaWYgbGVuKG1hdGNoZXMpICE9IDEgb3IgYW5zd2VyX3BhcnRbbWF0Y2hlc1swXS5lbmQoKTpdLnN0cmlwKCk6CiAgICAgICAgcmV0dXJuIFNjb3JlKFRydWUsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ21pc3Npbmdfb3JfZXh0cmFfZmluYWxfdGV4dCcpCiAgICB2YWx1ZSA9ICcgJy5qb2luKG1hdGNoZXNbMF0uZ3JvdXAoMSkubG93ZXIoKS5zcGxpdCgpKQogICAgaWYgbm90IHZhbHVlIG9yIGxlbih2YWx1ZSkgPiA4MDoKICAgICAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAnaW52YWxpZF9maW5hbF90ZXh0JykKICAgIGV4cGVjdGVkID0gJyAnLmpvaW4oc3RyKHRydXRoKS5sb3dlcigpLnNwbGl0KCkpCiAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgVHJ1ZSwgdmFsdWUgPT0gZXhwZWN0ZWQsIHZhbHVlLCAndmFsaWRfY29tcGxldGUnKQoKCmRlZiBkZWNvbXBvc2l0aW9uKGJhc2VsaW5lLCBlZGl0ZWQpOgogICAgIiIiUGFpcmVkIHVuaXRzOiBlYWNoIHJvdyBoYXMgeydjb21wbGV0ZWQnOmJvb2wsJ2NvcnJlY3QnOmJvb2x9OyBzYW1lIGl0ZW0gb3JkZXIuIiIiCiAgICBpZiBsZW4oYmFzZWxpbmUpICE9IGxlbihlZGl0ZWQpIG9yIG5vdCBiYXNlbGluZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdub25lbXB0eSBlcXVhbC1sZW5ndGggcGFpcmVkIGRhdGEgcmVxdWlyZWQnKQogICAgYm90aCwgY2hhbmdlZCwgcjJ3LCB3MnIgPSAwLCAwLCAwLCAwCiAgICBmb3IgYiwgZSBpbiB6aXAoYmFzZWxpbmUsIGVkaXRlZCk6CiAgICAgICAgaWYgYlsnY29ycmVjdCddIGFuZCBub3QgYlsnY29tcGxldGVkJ10gb3IgZVsnY29ycmVjdCddIGFuZCBub3QgZVsnY29tcGxldGVkJ106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2NvcnJlY3QgYW5zd2VycyBjYW5ub3QgYmUgaW5jb21wbGV0ZScpCiAgICAgICAgZGlmZiA9IGludChlWydjb3JyZWN0J10pIC0gaW50KGJbJ2NvcnJlY3QnXSkKICAgICAgICBpZiBiWydjb21wbGV0ZWQnXSA9PSBlWydjb21wbGV0ZWQnXToKICAgICAgICAgICAgYm90aCArPSBkaWZmCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2hhbmdlZCArPSBkaWZmCiAgICAgICAgdzJyICs9IGRpZmYgPT0gMQogICAgICAgIHIydyArPSBkaWZmID09IC0xCiAgICByZXR1cm4gZGljdChuPWxlbihiYXNlbGluZSksIG5ldF90b3RhbD1ib3RoK2NoYW5nZWQsIGJvdGhfY29tcGxldGU9Ym90aCwKICAgICAgICAgICAgICAgIGNvbXBsZXRpb25fY2hhbmdlZD1jaGFuZ2VkLCB3cm9uZ190b19yaWdodD13MnIsIHJpZ2h0X3RvX3dyb25nPXIydywKICAgICAgICAgICAgICAgIGJhc2VsaW5lX2NvcnJlY3Q9c3VtKHhbJ2NvcnJlY3QnXSBmb3IgeCBpbiBiYXNlbGluZSksCiAgICAgICAgICAgICAgICBlZGl0ZWRfY29ycmVjdD1zdW0oeFsnY29ycmVjdCddIGZvciB4IGluIGVkaXRlZCkpCgoKZGVmIHBhaXJlZF9ib290c3RyYXBfY2koYmFzZWxpbmUsIGVkaXRlZCwgc2VlZD0yMDI2MTEyMiwgcmVwcz0xMDAwMCk6CiAgICAiIiJFeHBsb3JhdG9yeSBwZXJjZW50aWxlIENJIG9uIGZpeGVkIDQ4LWl0ZW0gcGFpcmVkIGFjY3VyYWN5IGRpZmZlcmVuY2UuIiIiCiAgICBpZiBsZW4oYmFzZWxpbmUpICE9IGxlbihlZGl0ZWQpIG9yIG5vdCBiYXNlbGluZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIHBhaXJpbmcnKQogICAgZGlmZnMgPSBbaW50KGVbJ2NvcnJlY3QnXSkgLSBpbnQoYlsnY29ycmVjdCddKSBmb3IgYiwgZSBpbiB6aXAoYmFzZWxpbmUsIGVkaXRlZCldCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBuID0gbGVuKGRpZmZzKQogICAgdmFsdWVzID0gc29ydGVkKHN1bShybmcuY2hvaWNlKGRpZmZzKSBmb3IgXyBpbiByYW5nZShuKSkvbiBmb3IgXyBpbiByYW5nZShyZXBzKSkKICAgIHJldHVybiAodmFsdWVzW2ludCgwLjAyNSpyZXBzKV0sIHZhbHVlc1ttaW4ocmVwcy0xLGludCgwLjk3NSpyZXBzKSldKQo=', 'sha256': 'ce0fe0b4633e9c3175889b763fe91fa25e231f5e0889d061e24ee28bd23649ed'}, 'ex0005c_prepare.py': {'source_b64': 'IiIiRVgwMDA1QyBwcm9zcGVjdGl2ZSBzcGxpdC1hbmQtaGFzaCBidWlsZGVyLiBSZXF1aXJlcyBmcm96ZW4gSHVnZ2luZyBGYWNlIGRhdGFzZXRzLgoKTm8gaGVsZC1vdXQgdGV4dCBpcyBwcmludGVkLiBBbGwgcHJlZmxpZ2h0IHNlbGVjdGlvbnMgYXJlIGRldGVybWluZWQgYnkgaWRlbnRpZmllcnMsCnNvdXJjZSByZXZpc2lvbnMgYW5kIHRoZSBmcm96ZW4gc2VlZCwgbmV2ZXIgYnkgbW9kZWwgcGVyZm9ybWFuY2UuCiIiIgppbXBvcnQgaGFzaGxpYiwganNvbiwgc3lzLCBwbGF0Zm9ybQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBleDAwMDVjX3Njb3JlciBpbXBvcnQgcmVmZXJlbmNlCgpTRUVEID0gJ0VYMDAwNUMtMjAyNi0xMS0yMi12MC4zLWZhbWlseS1zZWFsJwpNT0RFTCA9ICdRd2VuL1F3ZW4zLTAuNkInCk1PREVMX1JFViA9ICdjMTg5OWRlMjg5YTA0ZDEyMTAwZGIzNzBkODE0ODVjZGY3NWU0N2NhJwpHU01fUkVWID0gJzc0MDMxMmFkZDg4Zjc4MTk3OGMwNjU4ODA2YzU5YmMyODE1Yjk4NjYnClNZTUJPTElDX1JFViA9ICc5M2I1YjM3NThkOWQ5ODQxZmZlODFkNmNkMmFlMmIwMzA2ODViMDc4JwoKCmRlZiBoKG9iaik6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhvYmosIHNvcnRfa2V5cz1UcnVlLCBlbnN1cmVfYXNjaWk9RmFsc2UsIHNlcGFyYXRvcnM9KCcsJywgJzonKSkuZW5jb2RlKCd1dGYtOCcpKS5oZXhkaWdlc3QoKQoKCmRlZiBzb3J0X2lkcyhpZHMsIG5hbWVzcGFjZSk6CiAgICByZXR1cm4gc29ydGVkKGlkcywga2V5PWxhbWJkYSB4OiBoYXNobGliLnNoYTI1NihmJ3tTRUVEfXx7bmFtZXNwYWNlfXx7eH0nLmVuY29kZSgpKS5oZXhkaWdlc3QoKSkKCgpkZWYgbm9ybWFsaXplKHMpOgogICAgcmV0dXJuICcgJy5qb2luKHN0cihzKS5sb3dlcigpLnNwbGl0KCkpCgoKZGVmIGxvYWRfb25lKG5hbWUsIGNvbmZpZywgcmV2KToKICAgIGZyb20gZGF0YXNldHMgaW1wb3J0IGxvYWRfZGF0YXNldAogICAgb2JqID0gbG9hZF9kYXRhc2V0KG5hbWUsIG5hbWU9Y29uZmlnLCByZXZpc2lvbj1yZXYpCiAgICAjIFVzZSB0aGUgZG9jdW1lbnRlZCBzcGxpdCBpZiBhdmFpbGFibGU7IGZhaWwgcmF0aGVyIHRoYW4gc2lsZW50bHkgY29uY2F0ZW5hdGUuCiAgICBpZiAndGVzdCcgaW4gb2JqOiByZXR1cm4gbGlzdChvYmpbJ3Rlc3QnXSkKICAgIGlmICd0cmFpbicgaW4gb2JqIGFuZCBsZW4ob2JqKSA9PSAxOiByZXR1cm4gbGlzdChvYmpbJ3RyYWluJ10pCiAgICBpZiBsZW4ob2JqKSA9PSAxOiByZXR1cm4gbGlzdChuZXh0KGl0ZXIob2JqLnZhbHVlcygpKSkpCiAgICByYWlzZSBWYWx1ZUVycm9yKGYndW5leHBlY3RlZCBzcGxpdCBzdHJ1Y3R1cmUge25hbWV9OiB7bGlzdChvYmoua2V5cygpKX0nKQoKCmRlZiBwcmVzZXJ2YXRpb24oKToKICAgICIiIjYwIHN5bnRoZXRpYyBhbmQgc21hbGwgZmFjdHVhbCBpdGVtczsgZGV0ZXJtaW5pc3RpYyBrZXkuIEVhY2ggaGFzIG9uZSBjYW5vbmljYWwgYW5zd2VyLiIiIgogICAgZmFjdF9wYWlycyA9IFsKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgRnJhbmNlPycsICdQYXJpcycpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBKYXBhbj8nLCAnVG9reW8nKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgS2VueWE/JywgJ05haXJvYmknKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgQ2FuYWRhPycsICdPdHRhd2EnKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgSXRhbHk/JywgJ1JvbWUnKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgU3BhaW4/JywgJ01hZHJpZCcpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBFZ3lwdD8nLCAnQ2Fpcm8nKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgUGVydT8nLCAnTGltYScpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBHcmVlY2U/JywgJ0F0aGVucycpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBJbmRpYT8nLCAnTmV3IERlbGhpJyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIE5vcndheT8nLCAnT3NsbycpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBTd2VkZW4/JywgJ1N0b2NraG9sbScpLAogICAgICAgICgnV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBBdXN0cmFsaWE/JywgJ0NhbmJlcnJhJyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIFBvcnR1Z2FsPycsICdMaXNib24nKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgVGhhaWxhbmQ/JywgJ0Jhbmdrb2snKSwKICAgICAgICAoJ1doYXQgaXMgdGhlIGNhcGl0YWwgb2YgQ2hpbGU/JywgJ1NhbnRpYWdvJyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIFR1cmtleT8nLCAnQW5rYXJhJyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIEFyZ2VudGluYT8nLCAnQnVlbm9zIEFpcmVzJyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIE1vcm9jY28/JywgJ1JhYmF0JyksCiAgICAgICAgKCdXaGF0IGlzIHRoZSBjYXBpdGFsIG9mIEZpbmxhbmQ/JywgJ0hlbHNpbmtpJyksCiAgICBdCiAgICBncmFtbWFyID0gWwogICAgICAgICgnU2hlIF9fXyB0byBzY2hvb2wgZXZlcnkgbW9ybmluZy4gQ2hvb3NlOiB3YWxrIC8gd2Fsa3MuJywgJ3dhbGtzJyksCiAgICAgICAgKCdUaGV5IF9fXyBvbiB0aGUgYnVzIG5vdy4gQ2hvb3NlOiBpcyAvIGFyZS4nLCAnYXJlJyksCiAgICAgICAgKCdIZSBoYXMgX19fIHRoZSBsZXR0ZXIuIENob29zZTogd3JvdGUgLyB3cml0dGVuLicsICd3cml0dGVuJyksCiAgICAgICAgKCdXZSBfX18gcmVhZHkuIENob29zZTogYW0gLyBhcmUuJywgJ2FyZScpLAogICAgICAgICgnVGhpcyBpcyBfX18gdW1icmVsbGEuIENob29zZTogYSAvIGFuLicsICdhbicpLAogICAgICAgICgnVGhlIGNoaWxkcmVuIF9fXyBvdXRkb29ycy4gQ2hvb3NlOiBwbGF5cyAvIHBsYXkuJywgJ3BsYXknKSwKICAgICAgICAoJ1llc3RlcmRheSBJIF9fXyBhIGJvb2suIENob29zZTogcmVhZCAvIHJlYWRzLicsICdyZWFkJyksCiAgICAgICAgKCdUaGF0IHRyZWUgX19fIHRhbGwuIENob29zZTogaXMgLyBhcmUuJywgJ2lzJyksCiAgICAgICAgKCdNYXlhIGFuZCBUb20gX19fIGZyaWVuZHMuIENob29zZTogaXMgLyBhcmUuJywgJ2FyZScpLAogICAgICAgICgnSSBoYXZlIF9fXyBvcmFuZ2VzLiBDaG9vc2U6IG11Y2ggLyBtYW55LicsICdtYW55JyksCiAgICAgICAgKCdUaGUgY2F0IF9fXyBzbGVlcGluZy4gQ2hvb3NlOiBpcyAvIGFyZS4nLCAnaXMnKSwKICAgICAgICAoJ05laXRoZXIgYW5zd2VyIF9fXyBjb3JyZWN0LiBDaG9vc2U6IGlzIC8gYXJlLicsICdpcycpLAogICAgICAgICgnQSBwYWlyIG9mIHNob2VzIF9fXyBvbiB0aGUgZmxvb3IuIENob29zZTogaXMgLyBhcmUuJywgJ2lzJyksCiAgICAgICAgKCdUaGVyZSBfX18gdGhyZWUgbGFtcHMuIENob29zZTogaXMgLyBhcmUuJywgJ2FyZScpLAogICAgICAgICgnTXkgc2lzdGVyIF9fXyBjb2ZmZWUuIENob29zZTogbGlrZXMgLyBsaWtlLicsICdsaWtlcycpLAogICAgICAgICgnRWFjaCBzdHVkZW50IF9fXyBhIHBlbmNpbC4gQ2hvb3NlOiBoYXMgLyBoYXZlLicsICdoYXMnKSwKICAgICAgICAoJ0hlIHJhbiBfX18gdGhhbiBJIGRpZC4gQ2hvb3NlOiBmYXN0ZXIgLyBmYXN0ZXN0LicsICdmYXN0ZXInKSwKICAgICAgICAoJ1RoZXkgd2VyZSBfX18gbG91ZGx5LiBDaG9vc2U6IHRhbGsgLyB0YWxraW5nLicsICd0YWxraW5nJyksCiAgICAgICAgKCdJIF9fXyBmaW5pc2hlZCBub3cuIENob29zZTogaGF2ZSAvIGhhcy4nLCAnaGF2ZScpLAogICAgICAgICgnU2hlIHNwb2tlIF9fXy4gQ2hvb3NlOiBjbGVhciAvIGNsZWFybHkuJywgJ2NsZWFybHknKSwKICAgIF0KICAgICMgU2VsZi1jb250YWluZWQgc3ltYm9saWMgc3lsbG9naXNtczogbm8gbWF0aGVtYXRpY3Mgb3IgZXh0ZXJuYWwgZmFjdHMuCiAgICBsb2dpYyA9IFtdCiAgICBmb3IgaSBpbiByYW5nZSgyMCk6CiAgICAgICAgYSA9IGYnemF2aXtpfScKICAgICAgICBiID0gZidsdW1vcntpfScKICAgICAgICBjID0gZid0YXJpbntpfScKICAgICAgICBpZiBpICUgMiA9PSAwOgogICAgICAgICAgICBwcm9tcHQgPSBmJ0FsbCB7YX0gYXJlIHtifS4gQWxsIHtifSBhcmUge2N9LiBJZiBzb21ldGhpbmcgaXMgYSB7YX0sIG11c3QgaXQgYmUgYSB7Y30/IEFuc3dlciB5ZXMgb3Igbm8uJwogICAgICAgICAgICBhbnN3ZXIgPSAneWVzJwogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByb21wdCA9IGYnQWxsIHthfSBhcmUge2J9LiBTb21lIHtifSBhcmUge2N9LiBNdXN0IGV2ZXJ5IHthfSBiZSBhIHtjfT8gQW5zd2VyIHllcyBvciBuby4nCiAgICAgICAgICAgIGFuc3dlciA9ICdubycKICAgICAgICBsb2dpYy5hcHBlbmQoKHByb21wdCxhbnN3ZXIpKQogICAgY2F0ZWdvcmllcyA9IGRpY3QoZmFjdHVhbD1mYWN0X3BhaXJzLCBncmFtbWFyPWdyYW1tYXIsIGxvZ2ljPWxvZ2ljKQogICAgYXNzZXJ0IGFsbChsZW4odik9PTIwIGZvciB2IGluIGNhdGVnb3JpZXMudmFsdWVzKCkpCiAgICBkZXYsIGhvbGQgPSBbXSwgW10KICAgIGZvciBjYXRlZ29yeSwgcGFpcnMgaW4gY2F0ZWdvcmllcy5pdGVtcygpOgogICAgICAgIHJlY29yZHMgPSBbZGljdChpZD1mJ3ByZXNfe2NhdGVnb3J5fV97aTowMmR9JyxmYW1pbHk9Y2F0ZWdvcnksc291cmNlPSdzeW50aGV0aWNfcHJlc2VydmF0aW9uJywKICAgICAgICAgICAgICAgICAgICAgICAgcXVlc3Rpb249cSx0cnV0aD1hLGtpbmQ9J3RleHQnKSBmb3IgaSwocSxhKSBpbiBlbnVtZXJhdGUocGFpcnMpXQogICAgICAgIG9yZGVyZWQgPSBzb3J0ZWQocmVjb3JkcyxrZXk9bGFtYmRhIHI6IGgoW1NFRUQsclsnaWQnXV0pKQogICAgICAgIGRldi5leHRlbmQob3JkZXJlZFs6NF0pOyBob2xkLmV4dGVuZChvcmRlcmVkWzQ6XSkKICAgIGFzc2VydCBsZW4oZGV2KT09MTIgYW5kIGxlbihob2xkKT09NDgKICAgIHJldHVybiBkZXYsaG9sZAoKCmRlZiBidWlsZF9zcGxpdChnc21fdHJhaW4sIGdzbV90ZXN0LCBzeW1ib2xpYyk6CiAgICAjIEFwcGxlIG9yaWdpbmFsX2lkIG1hcHMgdG8gR1NNOEsgdGVzdCByb3cgaW5kZXggcGVyIG9mZmljaWFsIGRhdGFzZXQgY2FyZC4KICAgIGFzc2VydCBsZW4oZ3NtX3RyYWluKT4yNCBhbmQgbGVuKGdzbV90ZXN0KT4yNAogICAgZmFtaWxpZXM9ZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciByb3cgaW4gc3ltYm9saWM6CiAgICAgICAgZm9yIGtleSBpbiBbJ2lkJywnaW5zdGFuY2UnLCdvcmlnaW5hbF9pZCcsJ3F1ZXN0aW9uJywnYW5zd2VyJywnb3JpZ2luYWxfcXVlc3Rpb24nXToKICAgICAgICAgICAgaWYga2V5IG5vdCBpbiByb3c6IHJhaXNlIFZhbHVlRXJyb3IoZidtaXNzaW5nIEFwcGxlIGtleSB7a2V5fScpCiAgICAgICAgb3JpZz1pbnQocm93WydvcmlnaW5hbF9pZCddKQogICAgICAgIGlmIG5vdCAwPD1vcmlnPGxlbihnc21fdGVzdCk6IHJhaXNlIFZhbHVlRXJyb3IoJ0FwcGxlIG9yaWdpbmFsX2lkIG5vdCBpbiBHU004SyB0ZXN0JykKICAgICAgICBpZiBub3JtYWxpemUocm93WydvcmlnaW5hbF9xdWVzdGlvbiddKSAhPSBub3JtYWxpemUoZ3NtX3Rlc3Rbb3JpZ11bJ3F1ZXN0aW9uJ10pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQXBwbGUvR1NNOEsgc291cmNlIGZhbWlseSBtaXNtYXRjaDoge29yaWd9OyBzdG9wIGluc3RlYWQgb2YgZ3Vlc3NpbmcnKQogICAgICAgIGZhbWlsaWVzWyhzdHIocm93WydpZCddKSxvcmlnKV0uYXBwZW5kKHJvdykKICAgICMgT25lIGdlbmVyYXRlZCB2YXJpYW50IHBlciAyNCAqZGlmZmVyZW50KiBzb3VyY2UgZmFtaWx5IElEcy4KICAgIGZhbV9rZXlzPXNvcnRfaWRzKGxpc3QoZmFtaWxpZXMpLCdzeW1ib2xpY19mYW1pbHknKVs6MjRdCiAgICBpZiBsZW4oZmFtX2tleXMpIT0yNDogcmFpc2UgVmFsdWVFcnJvcignbm90IGVub3VnaCBzeW1ib2xpYyBmYW1pbGllcycpCiAgICBoZWxkX3N5bWJvbHM9W10KICAgIHVzZWRfZ3NtX2lkcz1zZXQoKQogICAgZm9yIHRlbXBsYXRlX2lkLG9yaWcgaW4gZmFtX2tleXM6CiAgICAgICAgaWYgb3JpZyBpbiB1c2VkX2dzbV9pZHM6IHJhaXNlIFZhbHVlRXJyb3IoJ3R3byBzZWxlY3RlZCBBcHBsZSB0ZW1wbGF0ZXMgc2hhcmUgb3JpZ2luYWwgR1NNOEsgZmFtaWx5JykKICAgICAgICB1c2VkX2dzbV9pZHMuYWRkKG9yaWcpCiAgICAgICAgcm93cz1zb3J0ZWQoZmFtaWxpZXNbKHRlbXBsYXRlX2lkLG9yaWcpXSxrZXk9bGFtYmRhIHI6aChbU0VFRCwnc3ltYm9saWNfaW5zdGFuY2UnLHN0cihyWydpbnN0YW5jZSddKV0pKQogICAgICAgIHJvdz1yb3dzWzBdCiAgICAgICAgaGVsZF9zeW1ib2xzLmFwcGVuZChkaWN0KGlkPWYnc3ltYm9saWM6e3RlbXBsYXRlX2lkfTp7cm93WyJpbnN0YW5jZSJdfScsZmFtaWx5PWYnb3JpZ2luYWxfZ3NtOGtfdGVzdDp7b3JpZ30nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgc291cmNlPSdhcHBsZS9HU00tU3ltYm9saWM6bWFpbicsIHF1ZXN0aW9uPXJvd1sncXVlc3Rpb24nXSwgdHJ1dGg9c3RyKHJlZmVyZW5jZShyb3dbJ2Fuc3dlciddKSksa2luZD0nbWF0aCcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvcmlnaW5hbF9nc21fdGVzdF9pZD1vcmlnLCB0ZW1wbGF0ZV9pZD10ZW1wbGF0ZV9pZCwgaW5zdGFuY2U9c3RyKHJvd1snaW5zdGFuY2UnXSkpKQogICAgIyBUaGUgb3JkaW5hcnkgR1NNOEsgdGVzdCBob2xkb3V0IGV4Y2x1ZGVzIGV2ZXJ5IHNlbGVjdGVkIEFwcGxlIHNvdXJjZSBmYW1pbHkuCiAgICBjYW5kaWRhdGVzPVtpIGZvciBpIGluIHJhbmdlKGxlbihnc21fdGVzdCkpIGlmIGkgbm90IGluIHVzZWRfZ3NtX2lkc10KICAgIG9yZGluYXJ5PXNvcnRfaWRzKGNhbmRpZGF0ZXMsJ2dzbThrX3Rlc3QnKVs6MjRdCiAgICBkZXZpZHM9c29ydF9pZHMobGlzdChyYW5nZShsZW4oZ3NtX3RyYWluKSkpLCdnc204a190cmFpbicpWzoyNF0KICAgIG1hdGhkZXY9W2RpY3QoaWQ9Zidnc204a190cmFpbjp7aX0nLGZhbWlseT1mJ2dzbThrX3RyYWluOntpfScsc291cmNlPSdvcGVuYWkvZ3NtOGs6bWFpbi90cmFpbicsCiAgICAgICAgICAgICAgICAgIHF1ZXN0aW9uPWdzbV90cmFpbltpXVsncXVlc3Rpb24nXSwgdHJ1dGg9c3RyKHJlZmVyZW5jZShnc21fdHJhaW5baV1bJ2Fuc3dlciddKSksa2luZD0nbWF0aCcpIGZvciBpIGluIGRldmlkc10KICAgIG1hdGhvcmRpbmFyeT1bZGljdChpZD1mJ2dzbThrX3Rlc3Q6e2l9JyxmYW1pbHk9Zidnc204a190ZXN0OntpfScsc291cmNlPSdvcGVuYWkvZ3NtOGs6bWFpbi90ZXN0JywKICAgICAgICAgICAgICAgICAgcXVlc3Rpb249Z3NtX3Rlc3RbaV1bJ3F1ZXN0aW9uJ10sIHRydXRoPXN0cihyZWZlcmVuY2UoZ3NtX3Rlc3RbaV1bJ2Fuc3dlciddKSksa2luZD0nbWF0aCcpIGZvciBpIGluIG9yZGluYXJ5XQogICAgYXNzZXJ0IGxlbihtYXRoZGV2KT09MjQgYW5kIGxlbihtYXRob3JkaW5hcnkpPT0yNCBhbmQgbGVuKGhlbGRfc3ltYm9scyk9PTI0CiAgICBhc3NlcnQgbGVuKHtyWydmYW1pbHknXSBmb3IgciBpbiBtYXRob3JkaW5hcnkgKyBoZWxkX3N5bWJvbHN9KSA9PSA0OAogICAgYXNzZXJ0IGxlbih7clsncXVlc3Rpb24nXSBmb3IgciBpbiBtYXRoZGV2ICsgbWF0aG9yZGluYXJ5ICsgaGVsZF9zeW1ib2xzfSkgPT0gNzIKICAgIHJldHVybiBtYXRoZGV2LCBtYXRob3JkaW5hcnksIGhlbGRfc3ltYm9scwoKCmRlZiB3cml0ZV92YXVsdChyb290LCBnc21fdHJhaW4sIGdzbV90ZXN0LCBzeW1ib2xpYyk6CiAgICByb290PVBhdGgocm9vdCk7IHJvb3QubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBkZXYsb3JkaW5hcnksc3RydWN0dXJhbD1idWlsZF9zcGxpdChnc21fdHJhaW4sZ3NtX3Rlc3Qsc3ltYm9saWMpCiAgICBwZGV2LHBob2xkPXByZXNlcnZhdGlvbigpCiAgICBoZWxkPW9yZGluYXJ5K3N0cnVjdHVyYWwrcGhvbGQKICAgICMgSGFzaCBjb250ZW50IHNvIGFjY2lkZW50YWwgaW4tbWVtb3J5L2RhdGEgZWRpdHMgY2FuIGJlIGRldGVjdGVkIGFmdGVyIHNlYWwuCiAgICAjIE9ubHkgc2VhbGVkIGV2YWx1YXRvciBtYXkgbG9hZCBob2xkb3V0X3ZhdWx0Lmpzb24gYWZ0ZXIgaW52ZXN0aWdhdG9yIGNvbW1pdG1lbnQuCiAgICBtYW5pZmVzdD1kaWN0KHByb3RvY29sPSdFWDAwMDVDX3YwLjMnLHN0YXR1cz0nUFJFRkxJR0hUX09OTFknLAogICAgICAgICAgICAgICAgICBzZWVkPVNFRUQsIG1vZGVsPU1PREVMLG1vZGVsX3JldmlzaW9uPU1PREVMX1JFViwKICAgICAgICAgICAgICAgICAgZ3NtOGtfcmV2aXNpb249R1NNX1JFVixzeW1ib2xpY19yZXZpc2lvbj1TWU1CT0xJQ19SRVYsCiAgICAgICAgICAgICAgICAgIGNvdW50cz17J21hdGhfZGV2JzoyNCwncHJlc2VydmF0aW9uX2Rldic6MTIsJ21hdGhfb3JkaW5hcnlfaGVsZCc6MjQsJ21hdGhfc3ltYm9saWNfaGVsZCc6MjQsJ3ByZXNlcnZhdGlvbl9oZWxkJzo0OH0sCiAgICAgICAgICAgICAgICAgIGRldmVsb3BtZW50X2lkcz1beFsnaWQnXSBmb3IgeCBpbiBkZXYrcGRldl0sCiAgICAgICAgICAgICAgICAgIGhlbGRvdXRfaWRzPVt4WydpZCddIGZvciB4IGluIGhlbGRdLAogICAgICAgICAgICAgICAgICBkZXZfZGF0YV9oYXNoPWgoZGV2K3BkZXYpLGhvbGRvdXRfZGF0YV9oYXNoPWgoaGVsZCkpCiAgICAocm9vdC8nZGV2ZWxvcG1lbnQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkZXYrcGRldixpbmRlbnQ9MixlbnN1cmVfYXNjaWk9RmFsc2UpKQogICAgKHJvb3QvJ2hvbGRvdXRfdmF1bHQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhoZWxkLGluZGVudD0yLGVuc3VyZV9hc2NpaT1GYWxzZSkpCiAgICAocm9vdC8nbWFuaWZlc3QuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhtYW5pZmVzdCxpbmRlbnQ9MikpCiAgICAjIFByaW50IG9ubHkgaWRlbnRpZmllcnMsIGNoZWNrc3VtcyBhbmQgY291bnRzLCBub3QgaGVsZC1vdXQgcXVlc3Rpb25zL2Fuc3dlcnMuCiAgICBwcmludCgnUFJFRkxJR0hUIFNFQUwgR0VORVJBVEVEOyBOTyBIT0xET1VUIEVWQUxVQVRJT04nKQogICAgcHJpbnQoanNvbi5kdW1wcyh7azptYW5pZmVzdFtrXSBmb3IgayBpbiBbJ2NvdW50cycsJ2Rldl9kYXRhX2hhc2gnLCdob2xkb3V0X2RhdGFfaGFzaCddfSxpbmRlbnQ9MikpCiAgICByZXR1cm4gbWFuaWZlc3QKCgpkZWYgbWFpbigpOgogICAgZnJvbSBkYXRhc2V0cyBpbXBvcnQgbG9hZF9kYXRhc2V0CiAgICBwcmludCgnTG9hZGluZyBzb3VyY2UtcGlubmVkIGRhdGFzZXQgdmVyc2lvbnM7IG5vIG1vZGVsIGluZmVyZW5jZScpCiAgICB0cmFpbj1saXN0KGxvYWRfZGF0YXNldCgnb3BlbmFpL2dzbThrJyxuYW1lPSdtYWluJyxzcGxpdD0ndHJhaW4nLHJldmlzaW9uPUdTTV9SRVYpKQogICAgdGVzdD1saXN0KGxvYWRfZGF0YXNldCgnb3BlbmFpL2dzbThrJyxuYW1lPSdtYWluJyxzcGxpdD0ndGVzdCcscmV2aXNpb249R1NNX1JFVikpCiAgICBzeW09bG9hZF9vbmUoJ2FwcGxlL0dTTS1TeW1ib2xpYycsJ21haW4nLFNZTUJPTElDX1JFVikKICAgIHdyaXRlX3ZhdWx0KCdleDAwMDVjX3NlYWwnLHRyYWluLHRlc3Qsc3ltKQoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzogbWFpbigpCg==', 'sha256': '3f1459fa3f80965ab1b1ace67f9d55f0b49ef8da9f1df215750f4d4ad15bc885'}, 'test_ex0005c_scorer.py': {'source_b64': 'aW1wb3J0IHVuaXR0ZXN0CmZyb20gZXgwMDA1Y19zY29yZXIgaW1wb3J0IHNjb3JlX21hdGgsIHNjb3JlX3RleHQsIHJlZmVyZW5jZSwgZGVjb21wb3NpdGlvbiwgbnVtZXJpYwoKY2xhc3MgU2NvcmVyVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfZ29sZChzZWxmKToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN0cihyZWZlcmVuY2UoJ1dvcmtcbiMjIyMgMSwyNTAuNScpKSwgJzEyNTAuNScpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTogcmVmZXJlbmNlKCdubyBtYXJrZWQgYW5zd2VyJykKICAgIGRlZiB0ZXN0X3RoaW5raW5nX3ZhbGlkKHNlbGYpOgogICAgICAgIHI9c2NvcmVfbWF0aCgnPHRoaW5rPndvcmtpbmc8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiAxLDIwMFxuJywgJzEyMDAnLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShyLmNvbXBsZXRlZCBhbmQgci5jb3JyZWN0KQogICAgZGVmIHRlc3RfZmFrZV9vcl9taXNzaW5nX3RoaW5rX2FuZF9hbnN3ZXJfbGluZShzZWxmKToKICAgICAgICBmb3IgcmVzcG9uc2UgaW4gKCc8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiA0JywgJzx0aGluaz5zdHVmZjwvdGhpbms+XG5GaW5hbCBhbnN3ZXI6XG40Jyk6CiAgICAgICAgICAgIHI9c2NvcmVfbWF0aChyZXNwb25zZSwnNCcsZmluaXNoZWQ9VHJ1ZSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCkKICAgIGRlZiB0ZXN0X3dyb25nX2lzX2NvbXBsZXRlKHNlbGYpOgogICAgICAgIHI9c2NvcmVfbWF0aCgnPHRoaW5rPng8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiAzJywgJzInLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3RfdGhpbmtpbmdfbm90X2Nsb3NlZChzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz53b3JraW5nIGZvcmV2ZXInLCAnNCcsIGZpbmlzaGVkPVRydWUpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3RfbGVuZ3RoX2N1dG9mZihzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz54PC90aGluaz5cbkZpbmFsIGFuc3dlcjogNCcsICc0JywgZmluaXNoZWQ9RmFsc2UpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3Rfbm9fbXVsdGlwbGVfYW5zd2VycyhzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz54PC90aGluaz5cbkZpbmFsIGFuc3dlcjogNFxuRmluYWwgYW5zd2VyOiA1JywgJzUnLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoci5jb21wbGV0ZWQpCiAgICBkZWYgdGVzdF9zdHJpY3RfbnVtZXJpYyhzZWxmKToKICAgICAgICBmb3IgYmFkIGluICgnMTIgZG9sbGFycycsJzIrMicsJzEvMicsJzEsMjMnLCdOYU4nLCc0LjAuMCcpOgogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOiBudW1lcmljKGJhZCkKICAgIGRlZiB0ZXN0X3RleHQoc2VsZik6CiAgICAgICAgcj1zY29yZV90ZXh0KCc8dGhpbms+eDwvdGhpbms+XG5GaW5hbCBhbnN3ZXI6IFBhcmlzJywgJ3BhcmlzJywgZmluaXNoZWQ9VHJ1ZSkKICAgICAgICBzZWxmLmFzc2VydFRydWUoci5jb21wbGV0ZWQgYW5kIHIuY29ycmVjdCkKICAgIGRlZiB0ZXN0X2RlY29tcG9zZShzZWxmKToKICAgICAgICBCPVsoMSwwKSwoMSwwKSwoMSwxKSwoMCwwKSwoMCwwKSwoMSwxKV0KICAgICAgICBFPVsoMSwxKSwoMSwxKSwoMSwwKSwoMSwxKSwoMSwxKSwoMCwwKV0KICAgICAgICBkPWRlY29tcG9zaXRpb24oW2RpY3QoY29tcGxldGVkPWJvb2woYyksY29ycmVjdD1ib29sKGEpKSBmb3IgYyxhIGluIEJdLAogICAgICAgICAgICAgICAgICAgICAgICBbZGljdChjb21wbGV0ZWQ9Ym9vbChjKSxjb3JyZWN0PWJvb2woYSkpIGZvciBjLGEgaW4gRV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgoZFsnbmV0X3RvdGFsJ10sZFsnYm90aF9jb21wbGV0ZSddLGRbJ2NvbXBsZXRpb25fY2hhbmdlZCddKSwgKDIsMSwxKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKChkWyd3cm9uZ190b19yaWdodCddLGRbJ3JpZ2h0X3RvX3dyb25nJ10pLCAoNCwyKSkKICAgIGRlZiB0ZXN0X2ludmFsaWRfcGFpcihzZWxmKToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOiBkZWNvbXBvc2l0aW9uKFt7J2NvcnJlY3QnOlRydWUsJ2NvbXBsZXRlZCc6RmFsc2V9XSwgW3snY29ycmVjdCc6RmFsc2UsJ2NvbXBsZXRlZCc6RmFsc2V9XSkKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzogdW5pdHRlc3QubWFpbigpCg==', 'sha256': '0273e9f4eff27fe01b0a074640d6010ce905b5b6b696f9881fb1c6528ddee79a'}, 'test_ex0005c_prepare.py': {'source_b64': 'aW1wb3J0IHVuaXR0ZXN0CmZyb20gZXgwMDA1Y19wcmVwYXJlIGltcG9ydCBidWlsZF9zcGxpdCxwcmVzZXJ2YXRpb24KCmNsYXNzIFNwbGl0VGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfcHJlc2VydmF0aW9uKHNlbGYpOgogICAgICAgIGRldixoZWxkPXByZXNlcnZhdGlvbigpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oZGV2KSwxMik7IHNlbGYuYXNzZXJ0RXF1YWwobGVuKGhlbGQpLDQ4KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNldCh4WydpZCddIGZvciB4IGluIGRlditoZWxkKSksNjApCiAgICBkZWYgdGVzdF9mYW1pbHlfcGFydGl0aW9uKHNlbGYpOgogICAgICAgIHRyYWluPVtkaWN0KHF1ZXN0aW9uPWYnVHJhaW4gcXVlc3Rpb24ge2l9JyxhbnN3ZXI9ZidTb2x1dGlvblxuIyMjIyB7aX0nKSBmb3IgaSBpbiByYW5nZSg0MCldCiAgICAgICAgdGVzdD1bZGljdChxdWVzdGlvbj1mJ1Rlc3QgcXVlc3Rpb24ge2l9JyxhbnN3ZXI9ZidTb2x1dGlvblxuIyMjIyB7aX0nKSBmb3IgaSBpbiByYW5nZSg4MCldCiAgICAgICAgc3ltYm9saWM9W2RpY3QoaWQ9aSxpbnN0YW5jZT0wLG9yaWdpbmFsX2lkPWkscXVlc3Rpb249ZidWYXJpYW50IHF1ZXN0aW9uIHtpfScsYW5zd2VyPWYnU29sdXRpb25cbiMjIyMge2krMX0nLCBvcmlnaW5hbF9xdWVzdGlvbj1mJ1Rlc3QgcXVlc3Rpb24ge2l9JykgZm9yIGkgaW4gcmFuZ2UoMzApXQogICAgICAgIGRldixvcmRpbmFyeSxzaGlmdD1idWlsZF9zcGxpdCh0cmFpbix0ZXN0LHN5bWJvbGljKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoKGxlbihkZXYpLGxlbihvcmRpbmFyeSksbGVuKHNoaWZ0KSksKDI0LDI0LDI0KSkKICAgICAgICBzZWxmLmFzc2VydFRydWUoc2V0KHNbJ29yaWdpbmFsX2dzbV90ZXN0X2lkJ10gZm9yIHMgaW4gc2hpZnQpLmlzZGlzam9pbnQoaW50KG9bJ2lkJ10uc3BsaXQoJzonKVsxXSkgZm9yIG8gaW4gb3JkaW5hcnkpKQogICAgICAgIHN5bWJvbGljWzBdWydvcmlnaW5hbF9xdWVzdGlvbiddPSdOT1QgdGhlIGNvcnJlc3BvbmRpbmcgcXVlc3Rpb24nCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTogYnVpbGRfc3BsaXQodHJhaW4sdGVzdCxzeW1ib2xpYykKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzp1bml0dGVzdC5tYWluKCkK', 'sha256': 'dc2b0516990bed984ee6d12b798aa090c8fb6d382ae4838a5a1640692ad3b01f'}}
source_dir=Path('/content/ex0005c_v03')
source_dir.mkdir(exist_ok=True)
for name,item in snapshots.items():
    data=base64.b64decode(item['source_b64'])
    assert hashlib.sha256(data).hexdigest()==item['sha256'], 'source integrity error'
    (source_dir/name).write_bytes(data)
print('Verified local source files:',list(snapshots))
sys.path.insert(0,str(source_dir))


In [ ]:
# 12 fully offline parser/split assertions. Fail closed.
import subprocess,sys
p=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(source_dir),'-v'],capture_output=True,text=True)
print(p.stdout);print(p.stderr)
assert p.returncode==0, 'STOP: offline unit tests failed'


In [ ]:
# Seal sources and split by family, before any Luna API calls.
# Prints SHA-256 counts only, never held-out questions/answers.
import os,json
from ex0005c_prepare import main as prepare
os.chdir('/content/ex0005c_v03')
prepare()
seal_root=source_dir/'ex0005c_seal'
manifest=json.loads((seal_root/'manifest.json').read_text())
assert manifest['counts']=={'math_dev':24,'preservation_dev':12,'math_ordinary_held':24,'math_symbolic_held':24,'preservation_held':48}
assert len(manifest['heldout_ids'])==96 and len(manifest['development_ids'])==36
print('Manifest hash:',__import__('hashlib').sha256((seal_root/'manifest.json').read_bytes()).hexdigest())
print('Holdout vault exists but MUST remain unopened until all treatment commitments.')


In [ ]:
# Load the exact healthy Qwen model. DO NOT edit weights in this preflight.
from transformers import AutoTokenizer, AutoModelForCausalLM
from ex0005c_prepare import MODEL, MODEL_REV
import torch,time,hashlib
started=time.monotonic()
tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=MODEL_REV)
model=AutoModelForCausalLM.from_pretrained(MODEL,revision=MODEL_REV,
              torch_dtype=torch.bfloat16,device_map='auto')
model.eval()
assert str(model.dtype)=='torch.bfloat16'
assert len(model.model.layers)>14
W=model.model.layers[14].mlp.down_proj.weight
assert W.ndim==2 and W.shape[1]>390
print('Healthy Qwen loaded, site shape=',tuple(W.shape),'elapsed_s=',round(time.monotonic()-started,1))
# NO column changes. No hidden holdouts loaded for model evaluation.


In [ ]:
# Frozen exact Qwen thinking-mode prompt, sampling, random seeds, and parser.
from ex0005c_scorer import score_math, score_text
from ex0005c_prepare import SEED
import hashlib,json,torch

def item_seed(identifier):
    return int.from_bytes(hashlib.sha256(f'{SEED}|sample|{identifier}'.encode()).digest()[:8],'big') % (2**63-1)

def construct_prompt(item):
    demand = 'a number, with no units or words' if item['kind']=='math' else 'a short word or phrase'
    return ('Solve the following problem carefully. You may reason before responding. '
            'Finish with exactly one line of the form Final answer: VALUE. '
            f'For VALUE, use {demand}. Do not add anything after that line.\n\n'
            + item['question'])

def evaluate_dev_one(item):
    """FOR DEVELOPMENT ITEMS ONLY. Never call on holdout_vault before commitment."""
    prompt=construct_prompt(item)
    messages=[{'role':'user','content':prompt}]
    chat=tokenizer.apply_chat_template(messages,tokenize=False,
                                       add_generation_prompt=True,enable_thinking=True)
    inputs=tokenizer(chat,return_tensors='pt').to('cuda')
    budget=3072 if item['kind']=='math' else 512
    chat_end=tokenizer.convert_tokens_to_ids('<|im_end|>')
    stops=sorted(set([tokenizer.eos_token_id,chat_end]))
    # Transformers 4.57.3: generate() does not accept generator=...
    # Fork the CPU and current CUDA RNG states so each item gets its own
    # reproducible sampling seed, without changing later items' RNG state.
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(item_seed(item['id']))
        with torch.inference_mode():
            output=model.generate(**inputs,max_new_tokens=budget,do_sample=True,
                        temperature=0.6,top_p=0.95,top_k=20,min_p=0.0,
                        eos_token_id=stops,pad_token_id=tokenizer.eos_token_id)
    ids=output[0][inputs['input_ids'].shape[1]:].tolist()
    finished=(bool(ids) and ids[-1] in stops)
    raw=tokenizer.decode(ids,skip_special_tokens=False)
    scorer=(score_math if item['kind']=='math' else score_text)
    result=scorer(raw,item['truth'],finished=finished)
    return {'id':item['id'],'kind':item['kind'],'tokens':len(ids),
            'output':raw,'score':result.json()}

# Only two development examples for smoke test. This is not G0 qualification.
dev=json.loads((seal_root/'development.json').read_text())
dev_math=[x for x in dev if x['kind']=='math']
smoke=[]
for item in dev_math[:2]:
    ts=time.monotonic();row=evaluate_dev_one(item);row['elapsed_seconds']=time.monotonic()-ts
    smoke.append(row)
    print('DEV SMOKE',row['id'],'tokens=',row['tokens'],
          'correct=',row['score']['correct'],'complete=',row['score']['completed'],
          'seconds=',round(row['elapsed_seconds'],1))
(source_dir/'g0_smoke.json').write_text(json.dumps(smoke,indent=2))
print('Only development prompts used. No holdout scoring.')


In [ ]:
# OPTIONAL: complete 24-item development baseline, required for G0 PASS.
# Explicitly set RUN_FULL_DEV=True if enough time remains in the Sunday window.
RUN_FULL_DEV=False
if RUN_FULL_DEV:
    import datetime
    assert datetime.datetime.now(ZoneInfo('America/Denver')).hour<20,'STOP: hard deadline'
    runs=[]; t0=time.monotonic()
    for item in dev_math:
        assert datetime.datetime.now(ZoneInfo('America/Denver')).hour<20,'STOP: hard deadline'
        row=evaluate_dev_one(item)
        runs.append(row)
    correct=sum(r['score']['correct'] for r in runs)
    complete=sum(r['score']['completed'] for r in runs)
    g0_behavior=(2<=correct<=20 and complete>=4)
    (source_dir/'g0_development.json').write_text(json.dumps(runs,indent=2))
    print('G0 math development only:',{'correct':correct,'completed':complete,'n':len(runs),
             'behavior_qualification':g0_behavior,'seconds':round(time.monotonic()-t0,1)})
    if not g0_behavior: print('STOP: cannot qualify baseline under predeclared G0 rule')
else:
    print('NOT RUN: full 24-item development baseline; G0 remains incomplete.')


In [ ]:
# Export ONLY the preflight evidence; no holdout vault and no benchmark answers.
# THIS EXPORT DOES NOT MEAN G0 PASSED. It permits offline inspection.
import zipfile,hashlib
out='/content/EX0005C_v03_PRE_FLIGHT_receipts.zip'
keep=['manifest.json','g0_smoke.json','g0_development.json']
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in keep:
        p=(seal_root/name) if name=='manifest.json' else (source_dir/name)
        if p.exists(): z.write(p,arcname=name)
print(out,'sha256',hashlib.sha256(Path(out).read_bytes()).hexdigest())
from google.colab import files
files.download(out)


## Stopping boundary

Stop here. This notebook does not offer the Luna surgery tools, optimizer, or held-out evaluation. A separate post-freeze controller must be prospectively qualified with identical data access and objective. **Do not open the holdout vault in the same chat as the surgeon.** If your environment, time, parser, or dataset checks fail, record **NOT RUN** or **INCONCLUSIVE**, not an invented pass.